# Espirales y oscilaciones con decaimiento

Frecuencias no enteras producen combinaciones que pueden no repetir exactamente. Si agregamos un factor $e^{-\alpha t}$, cada componente se apaga y en el plano complejo dibuja una espiral.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Checkbox, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True


def setup_complex_axis(ax, lim=2, title=None):
    ax.axhline(0, color="0.55", lw=1)
    ax.axvline(0, color="0.55", lw=1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel("Real")
    ax.set_ylabel("Imaginaria")
    if title:
        ax.set_title(title)

def arrow(ax, z, color="C0", label=None, origin=0+0j, width=0.006):
    ax.arrow(origin.real, origin.imag, z.real, z.imag,
             head_width=0.08, length_includes_head=True,
             color=color, width=width, label=label)

def plot_phasor_sum(amplitudes, freqs, phases, t=0.0, decay=None, ncycles=2):
    amplitudes = np.asarray(amplitudes, dtype=float)
    freqs = np.asarray(freqs, dtype=float)
    phases = np.asarray(phases, dtype=float)
    if decay is None:
        decay = np.zeros_like(amplitudes)
    decay = np.asarray(decay, dtype=float)
    z = amplitudes * np.exp(-decay*t) * np.exp(1j*(freqs*t + phases))
    ts = np.linspace(0, ncycles*2*np.pi, 900)
    wave = np.sum(amplitudes[:, None] * np.exp(-decay[:, None]*ts)
                  * np.exp(1j*(freqs[:, None]*ts + phases[:, None])), axis=0)
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
    setup_complex_axis(ax0, max(1.2, 1.2*np.sum(np.abs(amplitudes))), "Fasores")
    current = 0+0j
    for comp in z:
        arrow(ax0, comp, origin=current, color="C1")
        current += comp
    ax0.plot(wave.real, wave.imag, color="0.75", lw=1)
    ax0.scatter([current.real], [current.imag], color="C3")
    ax1.plot(ts, wave.real, color="C0", label="Re")
    ax1.plot(ts, wave.imag, color="C2", alpha=0.7, label="Im")
    ax1.axvline(t, color="0.2", ls="--", lw=1)
    ax1.set_xlabel("t")
    ax1.legend()
    plt.show()


In [ ]:
def inharmonic(A1=1, w1=1.0, A2=1, w2=1.37, A3=0.5, w3=2.11, t=4.0):
    plot_phasor_sum([A1, A2, A3], [w1, w2, w3], [0, 0, 0], t=t, ncycles=5)

def decaying(A1=1, w1=1.0, d1=0.03, A2=1, w2=1.4, d2=0.06, A3=0.5, w3=2.2, d3=0.10, t=4.0):
    plot_phasor_sum([A1, A2, A3], [w1, w2, w3], [0, 0, 0], decay=[d1, d2, d3], t=t, ncycles=5)

def bell_resynthesis():
    fs = 44100
    ts = np.arange(0, 8.0, 1/fs)
    amps = np.array([1.0, 0.7, 0.4, 0.25, 0.2, 0.07])
    freqs = np.array([296, 595, 696, 1149, 1716, 3096])
    decays = np.array([3.0, 2.1, 1.9, 1.0, 0.8, 0.8])
    components = amps[:, None] * np.sin(2*np.pi*freqs[:, None]*ts) * np.exp(-ts/decays[:, None])
    snd = components.sum(axis=0)
    plt.figure(figsize=(10, 3))
    plt.plot(ts[:fs], snd[:fs])
    plt.title("primer segundo de una campana sintetica")
    plt.xlabel("tiempo (s)")
    plt.show()
    return snd, fs

if WIDGETS_AVAILABLE:
    interact(inharmonic, A1=(0.0,2.0,0.05), w1=(0.0,3.0,0.05),
             A2=(0.0,2.0,0.05), w2=(0.0,3.0,0.05), A3=(0.0,2.0,0.05), w3=(0.0,3.0,0.05),
             t=(0.0, 10.0, 0.1))
    interact(decaying, A1=(0.0,2.0,0.05), w1=(0.0,3.0,0.05), d1=(0.0,0.2,0.005),
             A2=(0.0,2.0,0.05), w2=(0.0,3.0,0.05), d2=(0.0,0.2,0.005),
             A3=(0.0,2.0,0.05), w3=(0.0,3.0,0.05), d3=(0.0,0.2,0.005), t=(0.0, 10.0, 0.1))
else:
    inharmonic(); decaying()

bell_resynthesis()
